In [ ]:
%run "../../scripts/01-check_setup.ipynb"

In [ ]:
import glob
from pathlib import Path
import json
import pandas as pd

html_files = sorted(glob.glob("resources/agenda_*.html"))
# html_files = sorted(Path("resources").glob("agenda_*.html"))
print(f"Found {len(html_files)} file(s): {html_files}")

In [ ]:
rows = []
for file_path in html_files:
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    rows.append({"metadata": json.dumps({"file_name": file_path}), "content_html": html_content})
    print(f"  Loaded {len(html_content):,} chars from {file_path}")

df_event = pd.DataFrame(rows).convert_dtypes()
display(df_event.dtypes)
df_event

In [ ]:
from IPython.display import HTML

# Read HTML and display for metadata's {"file_name": "resources/agenda_recap_2026.html"}
html_output = f'<base href="resources/images">' + df_event['content_html'].iloc[0]
# content = str(
#     df_event.loc[
#         df_event['metadata'].apply(lambda m: json.loads(m)['file_name'] == 'resources/agenda_recap_2026.html'),
#         'content_html'
#     ].squeeze()
# )
# html_output = content.replace('<head>', '<head><base href="resources/">', 1)

display(HTML(html_output))


In [ ]:
import importlib
import config_events
importlib.reload(config_events)
from config_events import HANA_TABLE_NAME, HANA_SCHEMA_NAME

In [ ]:
from hana_ml.dataframe import create_dataframe_from_pandas

hdf_event_bronze = create_dataframe_from_pandas(
    connection_context=myconn,
    pandas_df=df_event,
    table_name=HANA_TABLE_NAME,
    schema=HANA_SCHEMA_NAME,
    force=True,
    object_type_as_bin=True,
    table_structure={"metadata": "NVARCHAR(5000)", "content_html": "NCLOB"}
)

In [ ]:
pd.set_option("max_colwidth", 256)
hdf_event_bronze.head(1).collect().T